In [2]:
import os
from dotenv import load_dotenv
from groq import Groq

In [ ]:
import os
import re
from dotenv import load_dotenv
from groq import Groq

# Carrega a API Key do .env
load_dotenv()

class GroqSkillAgent:
    def __init__(self, skill_file):
        self.client = Groq(api_key=os.getenv("GROQ_API_KEY"))
        self.skill_path = skill_file
        self.modelo = "llama-3.3-70b-versatile"

    def _ler_skill(self):
        """Lê o conteúdo bruto do arquivo .skill"""
        with open(self.skill_path, 'r', encoding='utf-8') as f:
            return f.read()

    def _limpar_resposta(self, texto):
        """Garante que retorne apenas o SQL puro, sem markdown"""
        return re.sub(r"```(?:sql)?\n?|```", "", texto).strip()

    def executar(self, pergunta, schema_dinamico):
        """Mapeia o arquivo .skill e envia para o Groq"""
        template = self._ler_skill()
        
        # Injeção dinâmica no template da skill
        prompt_final = template.replace("{{schema}}", schema_dinamico)\
                               .replace("{{pergunta}}", pergunta)

        try:
            completion = self.client.chat.completions.create(
                model=self.modelo,
                messages=[
                    {"role": "system", "content": "Você é um executor de skills técnicas."},
                    {"role": "user", "content": prompt_final}
                ],
                temperature=0.0, # Precisão máxima para SQL
            )
            
            sql_bruto = completion.choices[0].message.content
            return self._limpar_resposta(sql_bruto)
            
        except Exception as e:
            return f"-- Erro na execução da skill: {e}"


<>:52: SyntaxWarning: invalid escape sequence '\s'
<>:52: SyntaxWarning: invalid escape sequence '\s'
C:\Users\carol\AppData\Local\Temp\ipykernel_27080\4056738637.py:52: SyntaxWarning: invalid escape sequence '\s'
  agente = GroqSkillAgent("..\skills\prompt_agente.md")


--- SQL GERADO PELA SKILL ---
SELECT SUM(valor) 
FROM vendas 
WHERE data = CURRENT_DATE - INTERVAL '1 day';


In [7]:
schema = \
"""
CREATE EXTERNAL TABLE IF NOT EXISTS "analytics_web"."navegacao_eventos" (
  "event_date" DATE COMMENT 'Data do evento no formato YYYY-MM-DD',
  "event_timestamp" BIGINT COMMENT 'Timestamp exato em microssegundos da ocorrência do evento',
  "user_pseudo_id" STRING COMMENT 'ID anônimo do dispositivo/browser (equivalente ao cid do GA4)',
  "user_id" STRING COMMENT 'ID único do usuário autenticado no sistema',
  "client_type" STRING COMMENT 'Segmentação do cliente: B2B, B2C, VIP, Standard',
  "company_name" STRING COMMENT 'Nome da empresa associada ao cliente (para análises corporativas)',
  "device_category" STRING COMMENT 'Categoria do dispositivo: mobile, tablet, desktop',
  "mobile_brand_name" STRING COMMENT 'Marca do dispositivo móvel (ex: Apple, Samsung, Xiaomi)',
  "operating_system" STRING COMMENT 'Sistema operacional utilizado (ex: iOS, Android, Windows)',
  "browser" STRING COMMENT 'Navegador utilizado (ex: Chrome, Safari, Firefox)',
  "page_location" STRING COMMENT 'URL completa da página onde o evento ocorreu',
  "page_referrer" STRING COMMENT 'URL da página anterior que levou o usuário à página atual',
  "page_title" STRING COMMENT 'Título da página (tag <title>) no momento do acesso',
  "session_id" STRING COMMENT 'ID único da sessão gerado no início da navegação',
  "engagement_time_msec" BIGINT COMMENT 'Tempo de engajamento na página em milissegundos'
)
COMMENT 'Tabela de rastreamento de navegação simulando estrutura GA4 para análise B2B/B2C'
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
LOCATION 's3://seu-bucket-de-dados/navegacao/';
"""

In [9]:

# --- Exemplo de Uso ---

if __name__ == "__main__":
    # Instancia apontando para o seu arquivo .skill
    agente = GroqSkillAgent("..\skills\prompt_agente.md")

    # Dados dinâmicos que podem vir de um banco ou API
    minha_pergunta = "Compare o tempo médio de engajamento entre clientes VIP e Standard no desktop durante o mês de maio, agrupado por device, em uma tabela pivotada para comparar os valores entre os meses"

    # Roda o agente
    resultado_sql = agente.executar(minha_pergunta, schema)

    print("--- SQL GERADO PELA SKILL ---")
    print(resultado_sql)

<>:5: SyntaxWarning: invalid escape sequence '\s'
<>:5: SyntaxWarning: invalid escape sequence '\s'
C:\Users\carol\AppData\Local\Temp\ipykernel_27080\2318427114.py:5: SyntaxWarning: invalid escape sequence '\s'
  agente = GroqSkillAgent("..\skills\prompt_agente.md")


--- SQL GERADO PELA SKILL ---
SELECT 
  EXTRACT(MONTH FROM event_date) AS mes,
  device_category,
  client_type,
  AVG(engagement_time_msec) AS tempo_medio_engajamento
FROM 
  analytics_web.navegacao_eventos
WHERE 
  device_category = 'desktop' 
  AND client_type IN ('VIP', 'Standard') 
  AND EXTRACT(MONTH FROM event_date) = 5
GROUP BY 
  EXTRACT(MONTH FROM event_date),
  device_category,
  client_type
PIVOT (
  AVG(engagement_time_msec) 
  FOR client_type IN ('VIP', 'Standard')
);
